##AI Knowledge Graph Builder for Enterprise Intelligence

#ISRO Mission Navigator

##MODULE 3: Graph Construction & Storage Hub

1.   Neo4j (Colab Docker + AuraDB)
2.   TigerGraph (Cloud)
1.   Your existing file:/triples.csv



####Step 1: Install Required Libraries

In [1]:
# Install Neo4j Python driver
!pip install neo4j

# Install Pandas
!pip install pandas

# Install NetworkX for visualization
!pip install networkx

# Install PyVis for interactive graphs
!pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.4 MB/s eta 0:00:00


####Step 2: Install and Start Docker (for Neo4j inside Colab)

In [2]:
!apt update
!apt install docker.io -y
!service docker start

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,776 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,910 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,767 kB]
Hit:11 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Ge

####Step 3: Run Neo4j Container

In [3]:
!docker run -d \
  --name neo4j \
  -p7474:7474 -p7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j

docker: Cannot connect to the Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?

Run 'docker run --help' for more information


####Step 4: Connect to Neo4j

In [4]:
from neo4j import GraphDatabase
import pandas as pd

uri = "neo4j+s://5be8df30.databases.neo4j.io"
username = "neo4j"
password = "jdrjnjNfgcFfBUY7PpsTE7YK92kLYXYlrCYZVOPcBiA"

driver = GraphDatabase.driver(uri, auth=(username, password))

In [5]:
#Load the Data from Module 2
df = pd.read_csv("/content/triples.csv")
df.head()

,Entity1,Relation,Entity2
0,GSAT-15,used_for,"Communication, Navigation"
1,GSLV-D5/GSAT-14,rocket_family,GSLV
2,CMS-01,used_for,Communication
3,CARTOSAT-2B,used_for,Earth Observation
4,INSAT-3B,orbit_type,GSO (Geosynchronous Orbit)


In [6]:
print(df.columns)

Index(['Entity1', 'Relation', 'Entity2'], dtype='object')


In [7]:
#Change the column name
df.columns = ['subject', 'relation', 'object']
print("Columns after cleaning:", df.columns)

Columns after cleaning: Index(['subject', 'relation', 'object'], dtype='object')


####Step 5: Graph Schema Creation

In [8]:
def create_constraints(tx):
    tx.run("""
        CREATE CONSTRAINT IF NOT EXISTS
        FOR (e:Entity)
        REQUIRE e.name IS UNIQUE
    """)

with driver.session() as session:
    session.execute_write(create_constraints)

print("Constraint Created Successfully ✅")

Constraint Created Successfully ✅


####Step 6: Insert Triples into Graph

In [9]:
#Core Insertion Function
def insert_triple(tx, subject, relation, obj):
    query = """
    MERGE (s:Entity {name: $subject})
    MERGE (o:Entity {name: $object})
    MERGE (s)-[r:RELATION {type: $relation}]->(o)
    """
    tx.run(query, subject=subject, relation=relation, object=obj)

In [10]:
#Bulk Insert from CSV
with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(
            insert_triple,
            row['subject'],
            row['relation'],
            row['object']
        )

print("Graph Successfully Created 🚀")

Graph Successfully Created 🚀


####Step 7: Verify Graph

In [11]:
#Count Total Nodes
def count_nodes(tx):
    result = tx.run("MATCH (n:Entity) RETURN COUNT(n) AS count")
    return result.single()["count"]

with driver.session() as session:
    total_nodes = session.execute_read(count_nodes)

print("Total Nodes:", total_nodes)

Total Nodes: 401


In [12]:
#Count Total Relationship
def count_relationships(tx):
    result = tx.run("MATCH ()-[r]->() RETURN COUNT(r) AS count")
    return result.single()["count"]

with driver.session() as session:
    total_rels = session.execute_read(count_relationships)

print("Total Relationships:", total_rels)

Total Relationships: 1257


In [13]:
# Check Unique Relationship Types
def get_relation_types(tx):
    query = """
    MATCH ()-[r]->()
    RETURN DISTINCT r.type AS relation_type
    """
    return [record["relation_type"] for record in tx.run(query)]

with driver.session() as session:
    relation_types = session.execute_read(get_relation_types)

print("Relation Types Found:", relation_types)

Relation Types Found: ['USED_FOR', 'LAUNCH_ON', 'ORBIT_TYPE', 'LAUNCHED_BY', 'used_for', 'launch_on', 'orbit_type', 'launched_by', 'ROCKET_FAMILY', 'rocket_family']


In [14]:
# Preview Sample Graph Data
def preview_graph(tx):
    query = """
    MATCH (a)-[r]->(b)
    RETURN a.name AS subject, r.type AS relation, b.name AS object
    LIMIT 10
    """
    return list(tx.run(query))

with driver.session() as session:
    preview = session.execute_read(preview_graph)

for record in preview:
    print(record)

<Record subject='GSAT-15' relation='USED_FOR' object='Communication, Navigation'>
<Record subject='GSAT-15' relation='LAUNCH_ON' object='2015-11-11'>
<Record subject='GSAT-15' relation='ORBIT_TYPE' object='GEO'>
<Record subject='GSAT-15' relation='LAUNCHED_BY' object='Ariane-5 VA-227'>
<Record subject='GSAT-15' relation='used_for' object='Communication, Navigation'>
<Record subject='GSAT-15' relation='launch_on' object='2015-11-11'>
<Record subject='GSAT-15' relation='orbit_type' object='GEO'>
<Record subject='GSAT-15' relation='launched_by' object='Ariane-5 VA-227'>
<Record subject='GSLV-D5/GSAT-14' relation='ROCKET_FAMILY' object='GSLV'>
<Record subject='GSLV-D5/GSAT-14' relation='rocket_family' object='GSLV'>


####Step 8: Improve Graph Model (Advanced Structure)

In [15]:
#Smart Label Assignment
def insert_triple_smart(tx, subject, relation, obj):

    if relation == "launched_by":
        query = """
        MERGE (s:Mission {name: $subject})
        MERGE (o:Rocket {name: $object})
        MERGE (s)-[r:LAUNCHED_BY]->(o)
        """
    else:
        query = """
        MERGE (s:Entity {name: $subject})
        MERGE (o:Entity {name: $object})
        MERGE (s)-[r:RELATION {type: $relation}]->(o)
        """

    tx.run(query, subject=subject, relation=relation, object=obj)

####Step 9: Incremental Updates (Production Method)

In [16]:
#Function to Add New Data
def update_graph_from_dataframe(new_df):
    new_df.columns = new_df.columns.str.strip().str.lower()

    with driver.session() as session:
        for index, row in new_df.iterrows():
            session.execute_write(
                insert_triple,
                row['subject'],
                row['relation'],
                row['object']
            )

    print("Graph Updated Successfully 🚀")

In [17]:
#Test Incremental Update
import pandas as pd

new_data = pd.DataFrame({
    "subject": ["Gaganyaan"],
    "relation": ["launched_by"],
    "object": ["LVM3"]
})

update_graph_from_dataframe(new_data)

Graph Updated Successfully 🚀


####Step 10: Graph Analytics (Enterprise Intelligence)

In [18]:
# Most Connected Node
def most_connected_node(tx):
    query = """
    MATCH (n)
    RETURN n.name AS node,
           COUNT { (n)--() } AS connections
    ORDER BY connections DESC
    LIMIT 1
    """
    return list(tx.run(query))

with driver.session() as session:
    result = session.execute_read(most_connected_node)

print(result)

[<Record node='PSLV' connections=102>]


In [19]:
# Most Used Rocket
def most_used_rocket(tx):
    query = """
    MATCH (m)-[:RELATION {type:'launched_by'}]->(r)
    RETURN r.name AS Rocket, COUNT(*) AS Launches
    ORDER BY Launches DESC
    """
    return list(tx.run(query))

with driver.session() as session:
    rockets = session.execute_read(most_used_rocket)

for r in rockets:
    print(r)

<Record Rocket='PSLV-C54/EOS-06 Mission' Launches=4>
<Record Rocket='PSLV-C40/Cartosat-2 Series Satellite Mission' Launches=3>
<Record Rocket='PSLV-C37 / Cartosat -2 Series Satellite' Launches=3>
<Record Rocket='PSLV-C16/RESOURCESAT-2' Launches=2>
<Record Rocket='PSLV-C9 / CARTOSAT \x96 2A' Launches=2>
<Record Rocket='Ariane-44L H10-3' Launches=2>
<Record Rocket='PSLV-C52/EOS-04 Mission' Launches=2>
<Record Rocket='PSLV-C6/CARTOSAT-1/HAMSAT' Launches=2>
<Record Rocket='PSLV-C7 / CARTOSAT-2 / SRE-1' Launches=2>
<Record Rocket='C-1 Intercosmos' Launches=2>
<Record Rocket='Vostok' Launches=2>
<Record Rocket='Ariane-5 VA-227' Launches=1>
<Record Rocket='PSLV-C50/CMS-01' Launches=1>
<Record Rocket='PSLV-C15/CARTOSAT-2B' Launches=1>
<Record Rocket='Ariane-5G' Launches=1>
<Record Rocket='SLV-3' Launches=1>
<Record Rocket='PSLV-D2' Launches=1>
<Record Rocket='Ariane5-V162' Launches=1>
<Record Rocket='PSLV-C19/RISAT-1' Launches=1>
<Record Rocket='Ariane-5 VA-215' Launches=1>
<Record Rocket='GSL

In [20]:
# Path Between Two Missions
def find_shortest_path(tx, start, end):
    query = """
    MATCH p = shortestPath(
        (a:Entity {name:$start})-[*]-
        (b:Entity {name:$end})
    )
    RETURN p
    """
    return list(tx.run(query, start=start, end=end))

with driver.session() as session:
    path = session.execute_read(find_shortest_path,
                                "Aryabhata",
                                "Chandrayaan-3")

print(path)

[<Record p=<Path start=<Node element_id='4:5d1673f2-e992-441e-b840-3f84f221de30:251' labels=frozenset({'Entity'}) properties={'name': 'Aryabhata'}> end=<Node element_id='4:5d1673f2-e992-441e-b840-3f84f221de30:103' labels=frozenset({'Satellite', 'Entity'}) properties={'name': 'Chandrayaan-3'}> size=8>>]


####Step 11: Export Graph Back to CSV

In [21]:
def export_graph(tx):
    query = """
    MATCH (a)-[r]->(b)
    RETURN a.name AS subject, r.type AS relation, b.name AS object
    """
    return list(tx.run(query))

with driver.session() as session:
    data = session.execute_read(export_graph)

export_df = pd.DataFrame(data)
export_df.to_csv("exported_graph.csv", index=False)
print("File Succesfully Created.")

File Succesfully Created.


####Step 12: Graph Performance Optimization

In [22]:
def create_index(tx):
    tx.run("CREATE INDEX entity_name IF NOT EXISTS FOR (e:Entity) ON (e.name)")

with driver.session() as session:
    session.execute_write(create_index)

####Step 13: Connect to Neo4j AuraDB (Cloud Production)

In [23]:
uri = "neo4j+s://5be8df30.databases.neo4j.io"
username = "neo4j"
password = "jdrjnjNfgcFfBUY7PpsTE7YK92kLYXYlrCYZVOPcBiA"

driver = GraphDatabase.driver(uri, auth=(username, password))